# Week 59, Day 1: Cross-validation and nested cross-validation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w059/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def population(n, g, p=50, signal=0.25):
    """n trials of p neurons; the stimulus (+1 or -1) shifts every neuron's rate a little, along a random direction."""
    w = np.random.default_rng(0).normal(size=p)
    y = np.where(g.random(n) < 0.5, 1.0, -1.0)
    X = signal * np.outer(y, w) + g.normal(size=(n, p))
    return X, y

def ridge_fit(X, y, lam):
    mu = X.mean(0)
    Xc = X - mu
    w = np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ (y - y.mean()))
    return w, mu, y.mean()

def ridge_acc(model, X, y):
    w, mu, b = model
    return np.mean(np.sign((X - mu) @ w + b) == y)

print("ready: population(n, g) simulates trials; ridge_fit and ridge_acc fit and score a ridge decoder")

### Exercise: K-fold from scratch

Write kfold_indices(n, K, g), which shuffles the trial indices and splits them into K nearly equal test folds, and cv_acc(X, y, lam, K, g), which returns the mean test accuracy across folds. Then compare, over 40 reshuffles of the same 120 trials, the spread of a single 75/25 split and of 5-fold cross-validation.

In [ ]:
def kfold_indices(n, K, g):
    return ...

def cv_acc(X, y, lam, K, g):
    scores = []
    for te in kfold_indices(len(y), K, g):
        tr = np.setdiff1d(np.arange(len(y)), te)
        scores.append(ridge_acc(ridge_fit(X[tr], y[tr], lam), X[te], y[te]))
    return np.mean(scores)

gk = np.random.default_rng(591)
X, y = population(120, gk)
single, fivefold = [], []
for s in range(40):
    perm = gk.permutation(120)
    tr, te = perm[:90], perm[90:]
    single.append(ridge_acc(ridge_fit(X[tr], y[tr], 10.0), X[te], y[te]))
    fivefold.append(cv_acc(X, y, 10.0, 5, gk))
print(f"single split: mean {np.mean(single):.3f}, spread (sd) {np.std(single):.3f}")
print(f"5-fold CV:    mean {np.mean(fivefold):.3f}, spread (sd) {np.std(fivefold):.3f}")

In [ ]:
import numpy as np
def test_folds():
    "kfold_indices returns K disjoint test folds that together cover every trial once"
    folds = kfold_indices(23, 5, np.random.default_rng(0))
    assert len(folds) == 5 and sorted(np.concatenate(folds).tolist()) == list(range(23))
def test_spread():
    "single 25% splits vary more across reshuffles than 5-fold estimates do"
    assert np.std(single) > np.std(fivefold)

test_folds()
test_spread()
print("All checks passed")

### Exercise: The winner's curse in decoding

For 20 simulated sessions of 80 trials, choose the penalty with the best 5-fold accuracy from the grid, record that best score, then refit with that penalty on all 80 trials and record its accuracy on 2,000 fresh trials.

In [ ]:
lams = np.logspace(-1, 4, 12)
go = np.random.default_rng(592)
X_new, y_new = population(2000, go, signal=0.15)
best_cv, fresh = [], []
for session in range(20):
    Xs, ys = population(80, go, signal=0.15)
    scores = [cv_acc(Xs, ys, lam, 5, go) for lam in lams]
    k = ...
    best_cv.append(scores[k])
    fresh.append(...)
print(f"mean best CV score {np.mean(best_cv):.3f}; mean accuracy of the chosen decoders on fresh data {np.mean(fresh):.3f}")

In [ ]:
import numpy as np
def test_optimism():
    "averaged over sessions, the best CV score exceeds the chosen decoder's accuracy on fresh data by more than 0.02"
    assert len(best_cv) == 20 and np.mean(best_cv) > np.mean(fresh) + 0.02

test_optimism()
print("All checks passed")

### Exercise: Nested cross-validation

Write nested_acc(X, y, lams, g): for each of 5 outer folds, choose the penalty by 5-fold cross-validation on the outer training trials only, refit on them, and score on the outer test fold. Run it on the same 20 sessions and compare with the fresh-data accuracies.

In [ ]:
def nested_acc(X, y, lams, g):
    outer = []
    for te in kfold_indices(len(y), 5, g):
        tr = np.setdiff1d(np.arange(len(y)), te)
        inner = [cv_acc(X[tr], y[tr], lam, 5, g) for lam in lams]
        lam_best = ...
        outer.append(...)
    return np.mean(outer)

go = np.random.default_rng(592)
_ = population(2000, go, signal=0.15)          # replay the same random stream as before
nested = []
for session in range(20):
    Xs, ys = population(80, go, signal=0.15)
    for lam in lams: cv_acc(Xs, ys, lam, 5, go)   # (keeps the sessions identical to Step 3)
    nested.append(nested_acc(Xs, ys, lams, go))
print(f"best CV {np.mean(best_cv):.3f}, nested CV {np.mean(nested):.3f}, fresh data {np.mean(fresh):.3f}")

In [ ]:
import numpy as np
def test_nested():
    "unlike the best inner score, the nested estimate is not optimistic about fresh-data accuracy"
    assert len(nested) == 20 and np.mean(nested) < np.mean(fresh) + 0.005 < np.mean(best_cv)

test_nested()
print("All checks passed")